# Generate OpenICU concept YAML files from the RICU concept dictionary

This notebook generates OpenICU concept config files for these RICU source keys by default:

- `eicu`
- `eicu_demo`
- `miiv`
- `mimic_demo`

`mimic_Demo` is normalized to `mimic_demo` automatically. By default, the notebook generates **simple concepts only**.

## Important
All paths are configured as plain strings in the first code cell below. Usually you only edit that first code cell and then run the notebook from top to bottom.


In [1]:
from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path

import yaml

# Make notebook output easier to read.
pd = None
try:
    import pandas as pd
except Exception:
    pass

print("Python:", sys.version)

# =============================================================================
# USER CONFIGURATION — EDIT THESE STRINGS ONLY
# =============================================================================
# Important:
# - Use absolute paths whenever possible.
# - Keep paths as strings here. They are converted to pathlib.Path below.
# - Empty string "" means: disabled / not used.

# Folder that contains: src/openicu_ricu_converter
CONVERTER_PROJECT_STR = "/home/q039tl/workspace/openicu_ricu_converter_project"

# RICU JSON concept dictionary file.
RICU_CONCEPT_DICT_STR = "/home/q039tl/workspace/ricu/inst/extdata/config/concept-dict.json"

# Optional existing OpenICU config folder. This helps infer event names.
# Set to "" if you do not want to use an existing OpenICU config.

# Output folder for the generated YAML config files.
# You said you now use ~/output for generated outputs/results, so this defaults there.
OUTPUT_CONFIG_STR = "/home/q039tl/output/generated_ricu_config_all_sources"


# Source keys requested for generation.
SOURCES_RAW = ["eicu", "eicu_demo", "miiv", "mimic_Demo"]

# Simple concepts only by default.
INCLUDE_DERIVED = False
COMPLEX_STUBS = False
OVERWRITE = True

# Optional filters. Keep None to generate all available concepts.
CONCEPTS = None       # example: ["dbp", "alb", "abx"]
CATEGORIES = None     # example: ["vitals", "chemistry", "hematology"]

# Optional final copy step. Default is safe and does nothing.

# =============================================================================
# DO NOT EDIT BELOW UNLESS YOU WANT TO CHANGE NOTEBOOK LOGIC
# =============================================================================

def as_required_path(value: str, name: str) -> Path:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} must be a non-empty path string.")
    return Path(value).expanduser().resolve()


def as_optional_path(value: str) -> Path | None:
    if not isinstance(value, str) or not value.strip():
        return None
    return Path(value).expanduser().resolve()


CONVERTER_PROJECT = as_required_path(CONVERTER_PROJECT_STR, "CONVERTER_PROJECT_STR")
RICU_CONCEPT_DICT = as_required_path(RICU_CONCEPT_DICT_STR, "RICU_CONCEPT_DICT_STR")
OUTPUT_CONFIG = as_required_path(OUTPUT_CONFIG_STR, "OUTPUT_CONFIG_STR")

print("Configured paths:")
print("  CONVERTER_PROJECT:", CONVERTER_PROJECT)
print("  RICU_CONCEPT_DICT:", RICU_CONCEPT_DICT)
print("  OUTPUT_CONFIG:", OUTPUT_CONFIG)


Python: 3.13.13 (main, Apr  8 2026, 09:49:30) [GCC 13.3.0]
Configured paths:
  CONVERTER_PROJECT: /home/q039tl/workspace/openicu_ricu_converter_project
  RICU_CONCEPT_DICT: /home/q039tl/workspace/ricu/inst/extdata/config/concept-dict.json
  OPENICU_CONFIG: /home/q039tl/workspace/OpenICU/config
  OUTPUT_CONFIG: /home/q039tl/output/generated_ricu_config_all_sources
  TARGET_OPENICU_CONFIG: /home/q039tl/workspace/OpenICU.example/config


## 1. Load converter project

This uses `CONVERTER_PROJECT_STR` from the first code cell. The path must point to the folder containing `src/openicu_ricu_converter`.


In [2]:
if not (CONVERTER_PROJECT / "src" / "openicu_ricu_converter").is_dir():
    raise FileNotFoundError(
        "Converter project not found or invalid. Expected a folder containing "
        f"src/openicu_ricu_converter, but got:\n  {CONVERTER_PROJECT}\n\n"
        "Fix CONVERTER_PROJECT_STR in the first code cell."
    )

sys.path.insert(0, str(CONVERTER_PROJECT / "src"))

from openicu_ricu_converter import build_generation_plan, write_generation_plan
from openicu_ricu_converter.loader import load_concept_dict

print("Converter project:", CONVERTER_PROJECT)


Converter project: /home/q039tl/workspace/openicu_ricu_converter_project


## 2. Normalize options

This cell derives the converter options from the strings and switches set in the first code cell.


In [3]:
# If the configured OpenICU config folder is missing, disable it explicitly.

# Normalize user-facing spellings to converter source keys.
def normalize_source(source: str) -> str:
    value = source.strip().replace("-", "_").lower()
    aliases = {
        "mimic_demo": "mimic_demo",
        "mimicdemo": "mimic_demo",
        "eicudemo": "eicu_demo",
        "eicu_demo": "eicu_demo",
        "mimic_iv": "miiv",
        "mimiciv": "miiv",
    }
    return aliases.get(value, value)

SOURCES = [normalize_source(source) for source in SOURCES_RAW]

# Build converter settings path if the project provides one.
SETTINGS_PATH = CONVERTER_PROJECT / "config" / "settings.yaml"
if not SETTINGS_PATH.exists():
    SETTINGS_PATH = None


print("Sources:", SOURCES)
print("Settings path:", SETTINGS_PATH if SETTINGS_PATH is not None else "not used")


Sources: ['eicu', 'eicu_demo', 'miiv', 'mimic_demo']
Settings path: not used
OpenICU config for converter: /home/q039tl/workspace/OpenICU/config


## 3. Validate inputs and inspect the RICU dictionary

In [4]:
if not RICU_CONCEPT_DICT.exists():
    raise FileNotFoundError(
        f"RICU concept dictionary not found: {RICU_CONCEPT_DICT}\n"
        "Fix RICU_CONCEPT_DICT_STR in the first code cell."
    )

concept_dict = load_concept_dict(RICU_CONCEPT_DICT)
print("RICU concepts loaded:", len(concept_dict))

source_counts = Counter()
for concept in concept_dict.values():
    for source in concept.sources:
        source_counts[source] += 1

rows = [{"source": source, "concepts_with_source": count} for source, count in sorted(source_counts.items())]
if pd is not None:
    display(pd.DataFrame(rows))
else:
    for row in rows:
        print(row)

missing_requested_sources = [source for source in SOURCES if source not in source_counts]
if missing_requested_sources:
    print("Warning: these requested source keys are not present in the RICU concept dictionary:", missing_requested_sources)
    print("The converter can still write global concepts, but source-specific mappings may be empty for those keys.")


RICU concepts loaded: 121


,source,concepts_with_source
0,aumc,87
1,eicu,89
2,eicu_demo,89
3,hirid,76
4,miiv,89
5,mimic,91
6,mimic_demo,91
7,sic,66


## 4. Build and write the generation plan

In [5]:
plan = build_generation_plan(
    concept_dict_path=RICU_CONCEPT_DICT,
    sources=SOURCES,
    settings_path=SETTINGS_PATH,
    concepts=CONCEPTS,
    categories=CATEGORIES,
    include_derived=INCLUDE_DERIVED,
    complex_stubs=COMPLEX_STUBS,
    regex_prefix_mode="none",
    logical_columns_mode="preserve",
)

written_files = write_generation_plan(
    plan=plan,
    output_config=OUTPUT_CONFIG,
    overwrite=OVERWRITE,
)

print("Generated YAML payloads:", len(plan.files))
print("Written files:", len(written_files))
print("Unsupported/report entries:", len(plan.unsupported))
print("Output config:", OUTPUT_CONFIG)

Generated YAML payloads: 425
Written files: 420
Unsupported/report entries: 167
Output config: /home/q039tl/output/generated_ricu_config_all_sources


## 5. Summarize generated files

In [6]:
def classify_generated_path(path: Path) -> dict[str, str]:
    parts = path.parts
    if len(parts) >= 2 and parts[0] == "concept":
        return {
            "kind": "global",
            "dataset": "",
            "version": "",
            "category_or_folder": parts[1] if len(parts) > 1 else "",
            "file": parts[-1],
        }
    if len(parts) >= 5 and parts[0] == "dataset":
        return {
            "kind": "dataset",
            "dataset": parts[1],
            "version": parts[2],
            "category_or_folder": parts[3],
            "file": parts[-1],
        }
    return {"kind": "other", "dataset": "", "version": "", "category_or_folder": "", "file": str(path)}

summary_rows = [classify_generated_path(Path(f.path)) for f in plan.files]
summary_counter = Counter((row["kind"], row["dataset"], row["version"]) for row in summary_rows)
summary = [
    {"kind": kind, "dataset": dataset, "version": version, "files": count}
    for (kind, dataset, version), count in sorted(summary_counter.items())
]

if pd is not None:
    display(pd.DataFrame(summary))
else:
    for row in summary:
        print(row)

print("First 20 generated relative paths:")
for file in plan.files[:20]:
    print("-", file.path)

,kind,dataset,version,files
0,dataset,eicu-crd,2.0,85
1,dataset,eicu-demo,2.0,76
2,dataset,mimic-demo,3.1,83
3,dataset,mimic-iv,3.1,87
4,global,,,94


First 20 generated relative paths:
- concept/medications/antibiotics.yml
- dataset/eicu-crd/2.0/concept/antibiotics.yml
- dataset/eicu-demo/2.0/concept/antibiotics.yml
- dataset/mimic-iv/3.1/concept/antibiotics.yml
- dataset/mimic-demo/3.1/concept/antibiotics.yml
- concept/medications/vasopressin_rate.yml
- dataset/eicu-crd/2.0/concept/vasopressin_rate.yml
- dataset/eicu-demo/2.0/concept/vasopressin_rate.yml
- dataset/mimic-iv/3.1/concept/vasopressin_rate.yml
- dataset/mimic-demo/3.1/concept/vasopressin_rate.yml
- concept/demographics/patient_admission_type.yml
- dataset/eicu-crd/2.0/concept/patient_admission_type.yml
- dataset/mimic-iv/3.1/concept/patient_admission_type.yml
- concept/demographics/patient_age.yml
- dataset/eicu-crd/2.0/concept/patient_age.yml
- dataset/eicu-demo/2.0/concept/patient_age.yml
- dataset/mimic-iv/3.1/concept/patient_age.yml
- concept/chemistry/albumin.yml
- dataset/eicu-crd/2.0/concept/albumin.yml
- dataset/eicu-demo/2.0/concept/albumin.yml


## 6. Inspect unsupported entries

Unsupported entries are expected for RICU callbacks that the converter intentionally does not translate automatically.

In [7]:
unsupported_rows = []
for item in plan.unsupported[:50]:
    unsupported_rows.append({
        "concept": getattr(item, "concept_key", None),
        "source": getattr(item, "source", None),
        "reason": getattr(item, "reason", None),
        # Some converter versions do not expose `detail`.
        "detail": getattr(item, "detail", None),
    })

if pd is not None:
    display(pd.DataFrame(unsupported_rows))
else:
    for row in unsupported_rows:
        print(row)

report_file = OUTPUT_CONFIG / "ricu_unsupported.yml"
print("Unsupported report:", report_file if report_file.exists() else "not written")


,concept,source,reason,detail
0,abx,eicu,RICU callback preserved only in unsupported re...,None
1,abx,eicu,RICU callback preserved only in unsupported re...,None
2,abx,eicu_demo,RICU callback preserved only in unsupported re...,None
3,abx,eicu_demo,RICU callback preserved only in unsupported re...,None
4,abx,miiv,RICU callback preserved only in unsupported re...,None
5,abx,miiv,RICU callback preserved only in unsupported re...,None
6,abx,mimic_demo,RICU callback preserved only in unsupported re...,None
7,abx,mimic_demo,RICU callback preserved only in unsupported re...,None
8,adh_rate,eicu,RICU callback preserved only in unsupported re...,None
9,adh_rate,eicu_demo,RICU callback preserved only in unsupported re...,None


Unsupported report: /home/q039tl/output/generated_ricu_config_all_sources/ricu_unsupported.yml


## 7. Optional: copy generated config into an OpenICU config folder

Run this only if you really want to overwrite/copy generated files into another OpenICU config directory. The default cell is safe and does nothing.

In [8]:
    import shutil

    if not OUTPUT_CONFIG.exists():
        raise FileNotFoundError(f"Generated output does not exist: {OUTPUT_CONFIG}")


    for child in ["concept", "dataset", "ricu_unsupported.yml"]:
        src = OUTPUT_CONFIG / child
        if not src.exists():
            continue
        if src.is_dir():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

else:


COPY_TO_OPENICU_CONFIG is False; nothing copied.
